# OneStream Chatbot Architecture - Two-Sided NLP Pipeline

**Purpose:** Run the validated two-sided NLP architecture on the real OneStream ticket dataset and Docling-converted KB documents. The architecture was first validated on synthetic data in `onestream_poc.ipynb` (8/8 go-checks passed); this notebook applies the same pipeline to real data.

**Architecture:**
```
TICKETS                                  KB DOCUMENTS
   |                                          |
Preprocessing                          Preprocessing
   |                                          |
LDA topics                             LDA topics
   |                                          |
   +===== Gap analysis (TF-IDF) =====+
                       |
              Final routing taxonomy
                       |
              Classifier training (LR, NB)
                       |
              Top keywords per class -> Dify
```

**What changes vs. the POC:**
- Phase 0 loads from disk instead of generating synthetic data
- Alignment thresholds bumped to `STRONG=0.50, MULTI=0.30` (suited to a larger, noisier corpus)
- The final assessment block reframes documentation gaps and orphan KB content as **findings to action**, not validation pass/fail.

**Course references:** Jurafsky & Martin (2014); Manning & Schutze (2003); Blei et al. (2003) for LDA; Roder et al. (2015) for c_v coherence.


## Setup

Course-aligned NLP packages: `gensim` for LDA, `scikit-learn` for classifiers and TF-IDF, `nltk` available for production tokenisation/lemmatisation. The KB loader below assumes Docling output (.md or .txt). If you later add .pdf or .docx, extend `read_kb_file()`.


In [ ]:
# Install if running for the first time
# !pip install gensim scikit-learn pandas numpy nltk

import random
import re
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import (
    TfidfVectorizer, ENGLISH_STOP_WORDS
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, cross_val_predict
)
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
print('Setup complete')


## Phase 0 - Configuration and data loading

**Edit the CONFIG cell below to point at your local files.** The notebook expects:
- A tickets CSV with one row per ticket (text column + id column)
- A folder of Docling-converted KB documents (.md or .txt). The folder is scanned recursively; each file becomes one KB document, keyed by its filename stem.

**Reminder on data governance:** confirm with your manager and Maersk IT that the ticket text you load here has already been anonymised (Microsoft Presidio in your Phase 1) before running. This notebook does not re-anonymise.


In [ ]:
# ======================================
# CONFIG - EDIT THESE PATHS
# ======================================
TICKETS_CSV = '/Users/linusstamovyu/Downloads/OS/Helpdesk(in).csv'
TEXT_COL    = 'TicketIssue'
ID_COL      = 'TicketID'

KB_FOLDER   = '/Users/linusstamovyu/Downloads/OS/kb_docling/'

MIN_TICKET_CHARS = 10

# ======================================
# Ticket exclusion: change requests are not FAQ-answerable, route them
# elsewhere. Drops T4-style noise (CR submissions with date stamps).
# Edit patterns to match how your tenant tags CRs.
# ======================================
EXCLUDE_TICKET_PATTERNS = [
    r'\bchange[\s_]?request\b',
    r'\bCR[\-\s]?\d+\b',
    r'\btst\s+(server|environment|application)\b',
]

# ======================================
# Ticket text normalization: domain typos and filler words.
# Applied before preprocessing so LDA and the classifier see clean text.
# Empty string '' deletes the term. Extend as you find more.
# ======================================
TICKET_REPLACEMENTS = {
    r'\bmegered\b':       'merged',
    r'\bmegere\b':        'merge',
    r'\btst\b':           'test',
    r'\bpls\b':           'please',
    r'\bplz\b':           'please',
    r'\bthx\b':           'thanks',
    r'\bkindly\b':        '',
    r'\bplease\b':        '',
    r'\bdear\s+team\b':   '',
    r'\bhi\s+team\b':     '',
    r'\bhello\s+team\b':  '',
}

# ======================================
# Load and clean tickets
# ======================================
tickets_df = pd.read_csv(TICKETS_CSV)
assert TEXT_COL in tickets_df.columns, f'TEXT_COL {TEXT_COL!r} not in CSV columns: {list(tickets_df.columns)}'
assert ID_COL in tickets_df.columns, f'ID_COL {ID_COL!r} not in CSV columns: {list(tickets_df.columns)}'

tickets_df = tickets_df[[ID_COL, TEXT_COL]].rename(
    columns={TEXT_COL: 'text', ID_COL: 'id'}
)
tickets_df = tickets_df.dropna(subset=['text']).copy()
tickets_df['text'] = tickets_df['text'].astype(str).str.strip()
n_loaded = len(tickets_df)

# Drop change-request tickets
if EXCLUDE_TICKET_PATTERNS:
    cr_pattern = re.compile('|'.join(EXCLUDE_TICKET_PATTERNS), re.IGNORECASE)
    cr_mask = tickets_df['text'].str.contains(cr_pattern)
    n_cr = int(cr_mask.sum())
    tickets_df = tickets_df[~cr_mask].copy()
    print(f'Dropped {n_cr} change-request tickets')

# Apply text replacements (typo/filler normalization)
if TICKET_REPLACEMENTS:
    def normalize(text):
        out = text
        for pat, repl in TICKET_REPLACEMENTS.items():
            out = re.sub(pat, repl, out, flags=re.IGNORECASE)
        return re.sub(r'\s+', ' ', out).strip()
    tickets_df['text'] = tickets_df['text'].apply(normalize)

# Drop tickets that are too short after cleaning
tickets_df = tickets_df[tickets_df['text'].str.len() >= MIN_TICKET_CHARS]
tickets_df = tickets_df.reset_index(drop=True)

print(f'Loaded {n_loaded} tickets, kept {len(tickets_df)} after filtering and cleaning')
print(f'Median chars: {int(tickets_df["text"].str.len().median())}, '
      f'95th pct chars: {int(tickets_df["text"].str.len().quantile(0.95))}')
print(f'Sample row: id={tickets_df["id"].iloc[0]}, text={tickets_df["text"].iloc[0][:150]!r}')


In [ ]:
# ======================================
# KB exclusion: reference data files (metadata tables) dominate the corpus
# by sheer size and contaminate topic modeling. Drop by filename pattern.
# Edit if your tenant uses different naming.
# ======================================
EXCLUDE_KB_PATTERNS = [
    r'metadata',          # OTHER_Metadata_Accounts, OTHER_metadata_PL_BS, etc.
]

# ======================================
# Sanitize KB text: strip Docling artefacts (base64 image data, HTML tags,
# markdown image markers, table separator rows). These tokens were the
# source of false KB topic coherence in the first run (ivborw, allud,
# true, false, png appearing as topic words).
# ======================================
def sanitize_kb_text(text):
    # Base64 image data
    text = re.sub(r'data:image/[^;]+;base64,[A-Za-z0-9+/=\s]+', '', text)
    # Markdown image markers and Docling image placeholders
    text = re.sub(r'!\[.*?\]\(.*?\)', '', text)
    text = re.sub(r'<!==\s*image\s*==>', '', text, flags=re.IGNORECASE)
    # Raw HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # Markdown table separator rows like |===|===|===|
    text = re.sub(r'^\s*\|?[-:\s|]+\|?\s*$', '', text, flags=re.MULTILINE)
    # HTML/XML attribute leakage from Docling table conversion
    text = re.sub(r'\b(true|false|allud|ivborw\w*)\b', '', text, flags=re.IGNORECASE)
    # Long alphanumeric runs that are likely encoded data, not words
    text = re.sub(r'\b[A-Za-z0-9+/]{40,}\b', '', text)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def read_kb_file(path):
    suffix = path.suffix.lower()
    if suffix in {'.md', '.txt'}:
        return path.read_text(encoding='utf-8', errors='ignore')
    raise ValueError(f'Unsupported file type {suffix} for {path.name}')

kb_folder = Path(KB_FOLDER)
assert kb_folder.exists(), f'KB_FOLDER does not exist: {kb_folder}'
assert kb_folder.is_dir(), f'KB_FOLDER is not a directory: {kb_folder}'

kb_paths_all = sorted(
    p for p in kb_folder.rglob('*')
    if p.is_file() and p.suffix.lower() in {'.md', '.txt'}
)

# Apply exclusions
exclude_re = re.compile('|'.join(EXCLUDE_KB_PATTERNS), re.IGNORECASE) if EXCLUDE_KB_PATTERNS else None
kb_paths, excluded = [], []
for p in kb_paths_all:
    if exclude_re and exclude_re.search(p.name):
        excluded.append(p.name)
    else:
        kb_paths.append(p)
if excluded:
    print(f'Excluded {len(excluded)} KB files matching EXCLUDE_KB_PATTERNS:')
    for name in excluded:
        print(f'  - {name}')

kb_documents = {}
for p in kb_paths:
    raw = read_kb_file(p)
    text = sanitize_kb_text(raw)
    if len(text.strip()) < 50:
        print(f'  Skipping {p.name}: too short after sanitization ({len(text.strip())} chars)')
        continue
    key = p.stem
    if key in kb_documents:
        key = f'{p.parent.name}_{p.stem}'
    kb_documents[key] = text

print(f'\nLoaded {len(kb_documents)} KB documents from {kb_folder}')
if kb_documents:
    char_lens = [len(v) for v in kb_documents.values()]
    print(f'  Char length after sanitization - min: {min(char_lens):,}, '
          f'median: {int(np.median(char_lens)):,}, max: {max(char_lens):,}')
    total = sum(char_lens)
    print(f'  Total cleaned corpus: {total:,} chars '
          f'(was likely much larger before sanitization)')

assert len(kb_documents) >= 2, 'Need at least 2 KB documents to run topic modelling'


## Phase 1 - Preprocessing both corpora identically

Apply the same preprocessing pipeline to tickets and KB sections so they live in the same vocabulary space. This is critical for the gap analysis in Phase 4.

**Course techniques used (Lab 02, Lab 03):** lowercasing, punctuation removal, tokenization, stopword removal.

**In production:** add NLTK `WordNetLemmatizer` or spaCy lemmatization for better term normalization. We omit it here for portability (NLTK downloads may be blocked in some sandboxes).

In [ ]:
# Add domain-specific stopwords - terms too generic to discriminate intents
# in the OneStream domain (every ticket mentions 'entity', 'system', etc.)
domain_stops = {'entity', 'system', 'data', 'period'}
STOPWORDS = ENGLISH_STOP_WORDS.union(domain_stops)

def preprocess(text):
    """Lowercase, strip punctuation, tokenize, remove stopwords and short tokens."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    return tokens

ticket_tokens = [preprocess(t) for t in tickets_df['text']]
kb_tokens = {name: preprocess(doc) for name, doc in kb_documents.items()}

print(f"Ticket raw:    {tickets_df['text'].iloc[0]}")
print(f"Ticket tokens: {ticket_tokens[0]}")
print(f"\nKB raw (first 100 chars): {list(kb_documents.values())[0][:100].strip()}")
print(f"KB tokens (first 12):    {list(kb_tokens.values())[0][:12]}")

Ticket raw:    Workflow not running on preiod close for entity NA02
Ticket tokens: ['workflow', 'running', 'preiod', 'close']

KB raw (first 100 chars): Workflow Setup and Configuration. This guide covers how to configure
        consolidation
KB tokens (first 12):    ['workflow', 'setup', 'configuration', 'guide', 'covers', 'configure', 'consolidation', 'workflows', 'create', 'new', 'workflow', 'navigate']


## Phase 2 - LDA topic discovery on tickets

Apply LDA across `k = 4..8` and select the value with the highest **c_v coherence score**. Coherence measures how interpretable the discovered topics are - it correlates with human ratings of topic quality (Roder et al., 2015).

**Course technique:** Lab 06 covers LDA via Gensim with coherence-based k-selection.

**Expected outcome:** With 7 underlying ticket intents (5 primary + 2 multi-label), LDA should find ~6-8 topics. Coherence > 0.40 is meaningful, > 0.55 is strong.

In [ ]:
# Build the gensim dictionary and bag-of-words corpus
ticket_dict = corpora.Dictionary(ticket_tokens)
ticket_dict.filter_extremes(no_below=2, no_above=0.7)
ticket_corpus = [ticket_dict.doc2bow(t) for t in ticket_tokens]

# Sweep k and pick best by coherence
best_k_tickets, best_score_tickets, best_lda_tickets = None, -1, None
for k in [4, 5, 6, 7, 8]:
    lda = LdaModel(
        corpus=ticket_corpus, id2word=ticket_dict,
        num_topics=k, random_state=42,
        passes=15, iterations=100, alpha='auto',
    )
    cm = CoherenceModel(
        model=lda, texts=ticket_tokens,
        dictionary=ticket_dict, coherence='c_v'
    )
    score = cm.get_coherence()
    print(f"  k={k}: coherence={score:.3f}")
    if score > best_score_tickets:
        best_k_tickets, best_score_tickets, best_lda_tickets = k, score, lda

print(f"\nBest ticket LDA: k={best_k_tickets}, coherence={best_score_tickets:.3f}\n")
print("Discovered ticket topics:")
ticket_topics = {}
for tid in range(best_k_tickets):
    words = [w for w, _ in best_lda_tickets.show_topic(tid, topn=8)]
    ticket_topics[tid] = words
    print(f"  Topic T{tid}: {', '.join(words)}")

  k=4: coherence=0.534
  k=5: coherence=0.535
  k=6: coherence=0.545
  k=7: coherence=0.549
  k=8: coherence=0.550

Best ticket LDA: k=8, coherence=0.550

Discovered ticket topics:
  Topic T0: access, permission, denied, error, workflow, blocking, report, settings
  Topic T1: excel, load, monthly, fails, import, errors, validation, multiple
  Topic T2: view, user, cube, rights, permissions, access, corp, missing
  Topic T3: workflow, consolidation, run, execution, close, stuck, error, timeout
  Topic T4: report, failing, close, tax, rule, output, current, fails
  Topic T5: trial, file, balance, import, load, upload, tax, rejected
  Topic T6: tax, provision, wrong, calculation, computation, rule, current, incorrect
  Topic T7: dashboard, tax, correctly, showing, formula, deferred, values, returning


## Phase 3 - LDA topic discovery on KB sections

Same LDA process, applied independently to KB sections. We sweep `k = 3..6` since the KB has fewer documents than the ticket corpus.

**Note on small-corpus LDA:** With only 15 KB sections, LDA may merge topics that have weak signal. On the full real KB this is unlikely. The architecture compensates by also using document-level orphan detection in Phase 5.

In [ ]:
kb_names = list(kb_tokens.keys())
kb_token_lists = list(kb_tokens.values())

kb_dict = corpora.Dictionary(kb_token_lists)
kb_dict.filter_extremes(no_below=1, no_above=0.8)
kb_corpus = [kb_dict.doc2bow(t) for t in kb_token_lists]

best_k_kb, best_score_kb, best_lda_kb = None, -1, None
for k in [3, 4, 5, 6]:
    lda = LdaModel(
        corpus=kb_corpus, id2word=kb_dict,
        num_topics=k, random_state=42,
        passes=20, iterations=200, alpha='auto',
    )
    cm = CoherenceModel(
        model=lda, texts=kb_token_lists,
        dictionary=kb_dict, coherence='c_v'
    )
    score = cm.get_coherence()
    print(f"  k={k}: coherence={score:.3f}")
    if score > best_score_kb:
        best_k_kb, best_score_kb, best_lda_kb = k, score, lda

print(f"\nBest KB LDA: k={best_k_kb}, coherence={best_score_kb:.3f}\n")
print("Discovered KB topics:")
kb_topics = {}
for tid in range(best_k_kb):
    words = [w for w, _ in best_lda_kb.show_topic(tid, topn=8)]
    kb_topics[tid] = words
    print(f"  Topic K{tid}: {', '.join(words)}")

  k=3: coherence=0.337
  k=4: coherence=0.485
  k=5: coherence=0.570
  k=6: coherence=0.467

Best KB LDA: k=5, coherence=0.570

Discovered KB topics:
  Topic K0: workflow, database, consolidation, servers, configuration, report, server, application
  Topic K1: validation, errors, excel, load, import, verify, run, missing
  Topic K2: tax, cube, dashboard, view, display, reference, configuration, formula
  Topic K3: tax, rules, provision, close, computation, calculations, formula, rule
  Topic K4: import, balance, trial, workflow, backup, excel, template, files


## Phase 4 - Vocabulary gap analysis

This is the analytical heart of the architecture. We compute centroids for each ticket topic and each KB topic in a **joint TF-IDF space**, then build the **alignment matrix** of pairwise cosine similarities.

**What the alignment matrix tells us:**
- Cell `(Ti, Kj)` close to 1.0 - ticket topic i is well-served by KB topic j
- Row of all low values - ticket topic with no KB match (documentation gap)
- Column of all low values - KB topic nobody asks about (orphan content)
- Multiple high values in a row - multi-label routing needed

**Course techniques (Lab 04, Lab 05):** TF-IDF vectorization, cosine similarity.

In [ ]:
# Joint TF-IDF: fit on combined corpus so vectors share vocabulary
joint_vectorizer = TfidfVectorizer(
    tokenizer=lambda x: x.split(),
    preprocessor=lambda x: x,
    token_pattern=None,
    min_df=1,
)
all_corpus_strings = (
    [' '.join(t) for t in ticket_tokens]
    + [' '.join(t) for t in kb_token_lists]
)
joint_vectorizer.fit(all_corpus_strings)

# Assign each ticket to its dominant LDA topic
ticket_topic_assignments = []
for bow in ticket_corpus:
    dist = best_lda_tickets.get_document_topics(bow, minimum_probability=0)
    ticket_topic_assignments.append(max(dist, key=lambda x: x[1])[0])

# Centroid of each ticket topic in joint TF-IDF space
ticket_centroids = []
for tid in range(best_k_tickets):
    members = [' '.join(ticket_tokens[i])
               for i, ta in enumerate(ticket_topic_assignments) if ta == tid]
    if members:
        vecs = joint_vectorizer.transform(members)
        ticket_centroids.append(np.asarray(vecs.mean(axis=0)).flatten())
    else:
        ticket_centroids.append(np.zeros(len(joint_vectorizer.vocabulary_)))
ticket_centroids = np.array(ticket_centroids)

# Same for KB topics
kb_topic_assignments = []
for bow in kb_corpus:
    dist = best_lda_kb.get_document_topics(bow, minimum_probability=0)
    kb_topic_assignments.append(max(dist, key=lambda x: x[1])[0])

kb_topic_to_docs = {tid: [] for tid in range(best_k_kb)}
for i, ta in enumerate(kb_topic_assignments):
    kb_topic_to_docs[ta].append(kb_names[i])

kb_centroids = []
for tid in range(best_k_kb):
    members = [' '.join(kb_tokens[name]) for name in kb_topic_to_docs[tid]]
    if members:
        vecs = joint_vectorizer.transform(members)
        kb_centroids.append(np.asarray(vecs.mean(axis=0)).flatten())
    else:
        kb_centroids.append(np.zeros(len(joint_vectorizer.vocabulary_)))
kb_centroids = np.array(kb_centroids)

# The alignment matrix
alignment = cosine_similarity(ticket_centroids, kb_centroids)
align_df = pd.DataFrame(
    alignment,
    index=[f"T{i}" for i in range(best_k_tickets)],
    columns=[f"K{j}" for j in range(best_k_kb)],
)
print("Alignment matrix (rows = ticket topics, columns = KB topics):\n")
print(align_df.round(2).to_string())

Alignment matrix (rows = ticket topics, columns = KB topics):

      K0    K1    K2    K3    K4
T0  0.12  0.01  0.05  0.00  0.10
T1  0.10  0.32  0.05  0.05  0.24
T2  0.05  0.08  0.27  0.06  0.03
T3  0.38  0.08  0.02  0.02  0.19
T4  0.17  0.19  0.11  0.17  0.02
T5  0.04  0.15  0.04  0.07  0.41
T6  0.04  0.34  0.23  0.58  0.06
T7  0.02  0.20  0.40  0.31  0.01


## Phase 4b - Sentence-embedding alignment (semantic alternative to TF-IDF)

TF-IDF cosine in Phase 4 measures literal token overlap. On this corpus the ticket vocabulary (colloquial, abbreviated, first-person) and the KB vocabulary (formal corporate prose) overlap weakly, which caps alignment scores. The textbook fix (Manning & Schutze 2003, ch. 14 on lexical sparsity) is to move from sparse term-matching to dense semantic embeddings.

This block uses `sentence-transformers/all-MiniLM-L6-v2`, a small (80 MB, CPU-friendly) BERT-family encoder, to recompute the alignment matrix in 384-dim semantic space. The course brief explicitly permits pre-trained language models like BERT for the final project.

**Run requirement:** `pip install sentence-transformers` (this notebook will skip Phase 4b cleanly if the package is not installed). The first run downloads the model (~80 MB). Subsequent runs use the local cache.

**What to compare:** maximum cosine in the matrix (TF-IDF capped at 0.28 in the first run) and the proportion of ticket topics that find at least one KB match. If the embedding matrix produces materially higher scores, the remainder of the pipeline (Phase 5 routing) should be re-run against `alignment_embed` instead of `alignment`.


In [ ]:
# ======================================
# Phase 4b - Optional: sentence-transformer alignment
# Skips cleanly if sentence-transformers is not installed.
# ======================================
try:
    from sentence_transformers import SentenceTransformer
    HAVE_ST = True
except ImportError:
    HAVE_ST = False
    print('sentence-transformers not installed. Skip Phase 4b or run:')
    print('    pip install sentence-transformers')

if HAVE_ST:
    print('Loading all-MiniLM-L6-v2 (first run downloads ~80 MB)...')
    encoder = SentenceTransformer('all-MiniLM-L6-v2')

    # Build a representative text per ticket topic (concat up to N member tickets)
    # and per KB document (truncate to first ~5000 chars to fit encoder context)
    MAX_TICKETS_PER_TOPIC = 50
    KB_TRUNCATE_CHARS = 5000

    ticket_topic_texts = []
    for tid in range(best_k_tickets):
        members = [tickets_df['text'].iloc[i]
                   for i, ta in enumerate(ticket_topic_assignments) if ta == tid]
        rep = ' . '.join(members[:MAX_TICKETS_PER_TOPIC]) if members else ''
        ticket_topic_texts.append(rep[:8000])

    kb_doc_texts = [kb_documents[name][:KB_TRUNCATE_CHARS] for name in kb_names]

    print(f'Encoding {len(ticket_topic_texts)} ticket topics and {len(kb_doc_texts)} KB docs...')
    ticket_topic_embed = encoder.encode(ticket_topic_texts, show_progress_bar=False,
                                        normalize_embeddings=True)
    kb_doc_embed = encoder.encode(kb_doc_texts, show_progress_bar=False,
                                  normalize_embeddings=True)

    # Doc-level alignment matrix: ticket topics vs each KB document
    alignment_embed_doc = cosine_similarity(ticket_topic_embed, kb_doc_embed)
    align_embed_df = pd.DataFrame(
        alignment_embed_doc.round(2),
        index=[f'T{i}' for i in range(best_k_tickets)],
        columns=[name[:30] for name in kb_names],
    )
    print('\nSemantic alignment matrix (ticket topics x KB documents):')
    print(align_embed_df.to_string())

    print(f'\nMax similarity (TF-IDF Phase 4):    {alignment.max():.3f}')
    print(f'Max similarity (embeddings 4b):     {alignment_embed_doc.max():.3f}')
    print(f'Mean similarity (TF-IDF Phase 4):   {alignment.mean():.3f}')
    print(f'Mean similarity (embeddings 4b):    {alignment_embed_doc.mean():.3f}')

    # Recommended thresholds in embedding space (cosine of normalized vectors
    # has a different distribution than TF-IDF cosine)
    STRONG_EMBED = 0.45
    MULTI_EMBED  = 0.35
    print(f'\nSuggested embedding thresholds: STRONG={STRONG_EMBED}, MULTI={MULTI_EMBED}')
    print('To use these in Phase 5 routing, replace `alignment` with `alignment_embed_doc`')
    print('and `STRONG`/`MULTI` with `STRONG_EMBED`/`MULTI_EMBED` in cell 15.')


## Phase 5 - Final routing taxonomy

Apply decision rules to the alignment matrix to produce the routing taxonomy. By default this uses the TF-IDF alignment from Phase 4. If Phase 4b (sentence-transformer alignment) produced materially higher scores, swap `alignment` for `alignment_embed_doc` and update the thresholds accordingly (`STRONG_EMBED=0.45, MULTI_EMBED=0.35` is a reasonable starting point in embedding space).

**Decision rules:**
1. Ticket topic with multiple matches at cosine >= MULTI - multi-label routing
2. Ticket topic with exactly one match at cosine >= STRONG - single-label routing
3. Ticket topic with no match at cosine >= MULTI - **documentation gap** flagged for the content team
4. KB document with no ticket-topic similarity >= MULTI - **orphan content** flagged for archival or rewrite


In [ ]:
STRONG = 0.50  # match threshold for real-data run (per POC notebook recommendation)
MULTI = 0.30   # multi-label threshold for real-data run

routing_classes, gaps = [], []
for ti in range(best_k_tickets):
    matches = [(kj, alignment[ti, kj]) for kj in range(best_k_kb)
               if alignment[ti, kj] >= MULTI]
    matches.sort(key=lambda x: x[1], reverse=True)
    strong_matches = [m for m in matches if m[1] >= STRONG]

    if len(matches) > 1:
        routing_classes.append({
            'ticket_topic': ti, 'keywords': ticket_topics[ti],
            'kb_topics': [m[0] for m in matches], 'multi_label': True,
        })
    elif len(strong_matches) == 1:
        routing_classes.append({
            'ticket_topic': ti, 'keywords': ticket_topics[ti],
            'kb_topics': [strong_matches[0][0]], 'multi_label': False,
        })
    else:
        gaps.append({'ticket_topic': ti, 'keywords': ticket_topics[ti]})

# Document-level orphan detection (more robust on small KBs than topic-level)
kb_doc_vectors = joint_vectorizer.transform(
    [' '.join(kb_tokens[name]) for name in kb_names]
)
doc_to_ticket_sim = cosine_similarity(
    np.asarray(kb_doc_vectors.todense()), ticket_centroids
)
orphans = []
for i, name in enumerate(kb_names):
    max_sim = doc_to_ticket_sim[i].max()
    if max_sim < MULTI:
        orphans.append((name, max_sim))

print(f"Routing classes built: {len(routing_classes)}")
for rc in routing_classes:
    flag = " [MULTI-LABEL]" if rc['multi_label'] else ""
    kb_doc_names = []
    for kj in rc['kb_topics']:
        kb_doc_names.extend(kb_topic_to_docs[kj])
    print(f"  T{rc['ticket_topic']}: {', '.join(rc['keywords'][:5])}"
          f" -> KBs {rc['kb_topics']} ({len(kb_doc_names)} docs){flag}")

print(f"\nDocumentation gaps (no KB match): {len(gaps)}")
for g in gaps:
    print(f"  T{g['ticket_topic']}: {', '.join(g['keywords'][:5])}")

print(f"\nOrphan KB documents (no ticket match): {len(orphans)}")
for name, sim in orphans:
    print(f"  {name} (best similarity = {sim:.2f})")

Routing classes built: 5
  T1: excel, load, monthly, fails, import -> KBs [1, 4] (6 docs) [MULTI-LABEL]
  T3: workflow, consolidation, run, execution, close -> KBs [0] (5 docs)
  T5: trial, file, balance, import, load -> KBs [4] (4 docs)
  T6: tax, provision, wrong, calculation, computation -> KBs [3, 1, 2] (6 docs) [MULTI-LABEL]
  T7: dashboard, tax, correctly, showing, formula -> KBs [2, 3, 1] (6 docs) [MULTI-LABEL]

Documentation gaps (no KB match): 3
  T0: access, permission, denied, error, workflow
  T2: view, user, cube, rights, permissions
  T4: report, failing, close, tax, rule

Orphan KB documents (no ticket match): 3
  system_arch_servers (best similarity = 0.09)
  system_arch_database (best similarity = 0.03)
  system_arch_backup (best similarity = 0.00)


## Phase 6 - Classifier training

Train supervised classifiers on tickets, using the LDA-derived dominant topic as the label. The classifiers are NOT deployed to Dify directly - their purpose is to extract the **discriminative keywords per class** that will populate the Dify classifier node descriptions.

**Course techniques (Lab 04, Lab 05, Lab 07):**
- TF-IDF + Multinomial Naive Bayes (baseline)
- TF-IDF + Logistic Regression (primary - coefficients give keywords)

**Evaluation:** stratified 5-fold cross-validation, macro F1.

**Important:** these labels are LDA pseudo-labels, not human-validated ground truth. In production we recommend manual annotation of a 100-200 ticket sample to validate.

In [ ]:
y = np.array(ticket_topic_assignments)
X_text = [' '.join(t) for t in ticket_tokens]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

tfidf = TfidfVectorizer(min_df=2, ngram_range=(1, 2))
X = tfidf.fit_transform(X_text)

nb_scores = cross_val_score(MultinomialNB(), X, y, cv=cv, scoring='f1_macro')
lr_scores = cross_val_score(
    LogisticRegression(max_iter=1000, C=1.0), X, y, cv=cv, scoring='f1_macro'
)

print(f"TF-IDF + Naive Bayes:         macro F1 = {nb_scores.mean():.3f} "
      f"(+/- {nb_scores.std():.3f})")
print(f"TF-IDF + Logistic Regression: macro F1 = {lr_scores.mean():.3f} "
      f"(+/- {lr_scores.std():.3f})")

TF-IDF + Naive Bayes:         macro F1 = 0.652 (+/- 0.120)
TF-IDF + Logistic Regression: macro F1 = 0.713 (+/- 0.062)


In [ ]:
# Train final LR on full data and extract top keywords per class
# These keywords are what we put into Dify classifier node descriptions
lr_final = LogisticRegression(max_iter=1000, C=1.0)
lr_final.fit(X, y)
feature_names = np.array(tfidf.get_feature_names_out())

print("Top keywords per class (from LR coefficients):\n")
for ci, cls in enumerate(lr_final.classes_):
    top_idx = lr_final.coef_[ci].argsort()[-10:][::-1]
    top = feature_names[top_idx]
    print(f"  Class T{cls}: {', '.join(top)}")

Top keywords per class (from LR coefficients):

  Class T0: permission, denied, error, apac, access, permission denied, settings, permission settings, error opening, opening
  Class T1: excel, load, column, user locked, locked, excel load, validation, load fails, errors, fails
  Class T2: user, view, cube view, cube, rights, permissions, corp, access, missing rights, missing
  Class T3: workflow, consolidation, run, stuck, execution, workflow execution, consolidation workflow, running, execute, execute workflow
  Class T4: report, close, current, output, failing, tax rule, fails, rule, formatting, timeout
  Class T5: trial, file, trial balance, balance, import, upload, rejected, balance load, missing tax, balance file
  Class T6: provision, tax, tax provision, calculation, triggering, wrong, tax calculation, tax computation, computation, incorrect
  Class T7: dashboard, correctly, returning, formula, working, tax, tax formula, blank, tile, dashboard tile


In [ ]:
# Confusion matrix from cross-val predictions
# Shows WHERE classifier confuses topics - usually maps to LDA topic boundaries
y_pred = cross_val_predict(
    LogisticRegression(max_iter=1000, C=1.0), X, y, cv=cv
)
cm_matrix = confusion_matrix(y, y_pred)
cm_df = pd.DataFrame(
    cm_matrix,
    index=[f"T{i}" for i in range(best_k_tickets)],
    columns=[f"T{i}" for i in range(best_k_tickets)],
)
print("Confusion matrix (rows = true LDA topic, cols = predicted):\n")
print(cm_df.to_string())
print("\nDiagonal dominance indicates clean LDA topic boundaries.")
print("Off-diagonal cells reveal which intents users describe similarly.")

Confusion matrix (rows = true LDA topic, cols = predicted):

    T0  T1  T2  T3  T4  T5  T6  T7
T0   4   0   1   5   0   1   0   0
T1   0   8   0   1   0   2   0   2
T2   0   0  18   1   0   1   0   1
T3   0   0   1  29   0   0   0   0
T4   0   0   0   2   5   1   2   1
T5   0   1   1   0   1  16   0   0
T6   0   0   0   1   0   0  17   0
T7   0   0   0   0   0   1   3  12

Diagonal dominance indicates clean LDA topic boundaries.
Off-diagonal cells reveal which intents users describe similarly.


## Pipeline sanity checks and findings

On real data, documentation gaps and orphan KB content are not validation failures - they are the actionable output of the pipeline. This block separates two things:

**Sanity checks** (must pass to trust the run): coherence and classifier F1.

**Findings** (the deliverable): how many routing classes were built, how many documentation gaps surfaced, how many orphan KB documents were detected.


In [ ]:
# ======================================
# Sanity checks - must pass to trust the run
# ======================================
sanity = {
    'Ticket LDA coherence > 0.40':
        best_score_tickets > 0.40,
    'KB LDA coherence > 0.40':
        best_score_kb > 0.40,
    'Logistic Regression macro F1 > 0.55':
        lr_scores.mean() > 0.55,
    'LR matches or beats Naive Bayes':
        lr_scores.mean() >= nb_scores.mean() - 0.05,
}

print('Sanity checks:')
passed = 0
for check, ok in sanity.items():
    mark = 'PASS' if ok else 'FAIL'
    print(f'  [{mark}] {check}')
    passed += int(ok)
print(f'  -> {passed}/{len(sanity)} sanity checks passed')

# ======================================
# Findings - the actionable output
# ======================================
print('\nFindings:')
print(f'  Ticket topics discovered (k):     {best_k_tickets}')
print(f'  KB topics discovered (k):         {best_k_kb}')
print(f'  Routing classes built:            {len(routing_classes)}')
multi = sum(1 for rc in routing_classes if rc["multi_label"])
print(f'    of which multi-label:           {multi}')
print(f'  Documentation gaps (no KB match): {len(gaps)}')
print(f'  Orphan KB docs (no ticket demand):{len(orphans)}')

if passed == len(sanity):
    print('\nAll sanity checks passed. Routing taxonomy and gap/orphan')
    print('lists above are ready for stakeholder review.')
else:
    print('\nOne or more sanity checks failed. Inspect the offending')
    print('phase before treating the findings as deliverables.')


  [PASS] Ticket LDA coherence > 0.40
  [PASS] KB LDA coherence > 0.40
  [PASS] Number of ticket topics in expected range (4-8)
  [PASS] At least one routing class built
  [PASS] Documentation gap detected (Permissions intent)
  [PASS] Orphan KB detected (System Architecture)
  [PASS] Logistic Regression macro F1 > 0.55
  [PASS] LR outperforms or matches Naive Bayes

Result: 8/8 checks passed

GO -- the architecture works on synthetic data.
     Safe to scale to the full 1,800-ticket dataset.


## What this run produces and how to use it

**Outputs of this notebook:**
1. A list of `routing_classes`: each is a discovered ticket intent with its top LDA keywords and the KB topics that serve it. These map to Dify classifier nodes.
2. `gaps`: ticket intents with no KB coverage. These are documentation requests for the content team.
3. `orphans`: KB documents nobody asks about. Candidates for archival, consolidation, or rewrite.
4. Per-class top keywords from the Logistic Regression coefficients. These populate Dify classifier node descriptions.

**Recommended next steps:**
1. Manually annotate 100-200 tickets to validate the LDA pseudo-labels against human judgement before deploying.
2. Re-tune `STRONG` and `MULTI` thresholds against the real alignment matrix if the routing classes look too coarse or too fragmented.
3. Run a stakeholder workshop using the routing taxonomy and gap list as the facilitation artefact.
4. Consider BERTopic as an LDA alternative if coherence stays below 0.55 on the real corpus.
5. Push the keyword lists into Dify and route ambiguous queries (low softmax confidence) to a human agent.

**References:**
- Blei, D. M., Ng, A. Y., & Jordan, M. I. (2003). Latent Dirichlet Allocation. *Journal of Machine Learning Research*, 3, 993-1022.
- Jurafsky, D., & Martin, J. H. (2014). *Speech and Language Processing* (2nd ed.). Pearson.
- Manning, C. D., & Schutze, H. (2003). *Foundations of Statistical Natural Language Processing*. MIT Press.
- Roder, M., Both, A., & Hinneburg, A. (2015). Exploring the space of topic coherence measures. *Proceedings of WSDM 2015*.
